In [1]:
from CNN_NAS.ChildCNNModel import ChildCNNModel
# Some magic so that the notebook will reload the external python script file any time you edit and save the .py file;
%load_ext autoreload
%autoreload 2

([('64', '3', '1'), ('64', '3', '2'), ('128', '5', '1'), ('256', '5', '1'), ('256', '3', '2')], tensor([[-4.2107],
        [-3.1180],
        [-4.1659],
        [-3.6175],
        [-3.5022]], grad_fn=<StackBackward0>))
CNNController(
  (meta_layer): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=False)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=False)
  )
  (embedding): Embedding(36, 8)
  (rnn): LSTM(8, 32, batch_first=True)
  (fc_filter): Linear(in_features=32, out_features=4, bias=True)
  (fc_kernel): Linear(in_features=32, out_features=3, bias=True)
  (fc_padding): Linear(in_features=32, out_features=3, bias=True)
)


In [2]:
import torch
import torch.nn as nn
import time
from torch.utils.data import DataLoader
import os

import utils

import logging
logging.basicConfig(level=logging.INFO, filename=os.path.join(os.getcwd(), 'log.log'), filemode='w')

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

device = utils.get_device_available()
print(torch.__version__)
print(device)

2.4.1
cuda


In [3]:
def is_valid_encoding(encoding):
    if len(encoding) < 3 or encoding[0] != "START" or encoding[-1] != "END":
        return False

    for i in range(1, len(encoding) - 1):
        if not encoding[i].isnumeric():
            return False

    return True


def split_dataset(data, labels, split_ratio=0.8):
    dataset = torch.utils.data.TensorDataset(data, labels)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_data, train_labels = zip(*train_set)
    train_data = torch.stack(train_data)
    train_labels = torch.stack(train_labels)

    test_data, test_labels = zip(*test_set)
    test_data = torch.stack(test_data)
    test_labels = torch.stack(test_labels)

    return (train_data, train_labels), (test_data, test_labels)

### RL Loop

### CIFAR Dataset

In [30]:
# train_data = (torch.load(data_path + 'cifar/train_data.pt'), torch.load(data_path + 'cifar/train_label.pt'))
# test_data = (torch.load(data_path + 'cifar/test_data.pt'), torch.load(data_path + 'cifar/test_label.pt'))

# model_iters=1
# train_epochs=1
# trained_controller,optimizer = train_controller(CNNController(name="EXP_1_CIFAR", meta=[32, 10]), train_data, test_data, model_iters=model_iters, train_epochs=train_epochs, negative_reward=-1,
#                  max_child_layers=1)

# # save_controller(trained_controller,optimizer, f"CIFAR-10_MI{model_iters}_TE{train_epochs}_controller")

## CIFAR

In [4]:
dataset="cifar"

data_path = utils.check_cifar_dataset_exists()

dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + 'cifar/train_label.pt', weights_only=True))

dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))


num_channels = 1

if len(dataset_train_data.size())==4:
    num_channels=dataset_train_data.size(1)



num_classes = dataset_train_label.unique().size(0)
height = dataset_train_data.size(-2)
width = dataset_train_data.size(-1)

print(f"Height: {height}")
print(f"Width: {width}")
print(f"Number of channels: {num_channels}")
print(f"Number of classes:  {num_classes}")



Height: 32
Width: 32
Number of channels: 3
Number of classes:  10


## Load  predefined model encoding

In [8]:
import predefined_models
# Defined by data

base_model_encoding_dict = {}

base_model_encoding_dict["Benchmark_Model"] = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Lenet"] = predefined_models.get_lenet(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["VGG_11"] = predefined_models.get_vgg11(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Alexnet"] = predefined_models.get_alexnet(input_channels=num_channels, output_dim=num_classes)


# base_model = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)

# print(base_model)

[[[3, 256, 3, 1], [2, 2], [256, 128, 3, 1], [2, 2], [128, 64, 3, 1], [2, 2]], [[1024, 512], [512, 256], [256, 10]]]


In [9]:
from CNN_NAS.CNNController import CNNController

#Load and run
logger.info("###############################################")
logger.info("STARTING TRAINING")
logger.info("###############################################")


# print(model)

for base_model in base_model_encoding_dict:
    model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
    name = base_model
    total_epochs=0
    for i in range(3):
        model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=10)
        test_accuracy = model.evaluate_model(data=dataset_test_data,labels=dataset_test_label)
        total_epochs +=10
        print(f"Model {name} Test Accuracy: {test_accuracy} with total epochs {total_epochs}")

ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): Linear(in_features=1024, out_features=512, bias=True)
    (11): ReLU()
    (12): Linear(in_features=512, out_features=256, bias=True)
    (13): ReLU()
    (14): Linear(in_features=256, out_features=10, bias=True)
  )
)
Test Accuracy: 0.7458
